# Pakistan Housing Market — Exploratory Data Analysis

An in-depth exploration of **16,000+ property listings** from [Zameen.com](https://www.zameen.com/), Pakistan's largest real estate marketplace. This analysis examines pricing patterns, geographic trends, and property characteristics across 12 major Pakistani cities to uncover actionable insights for real estate investors and market analysts.

**Dataset:** Zameen.com Housing Prices (Kaggle)  
**Tools:** Python, pandas, matplotlib, seaborn, plotly

## 1. Setup & Data Loading

In [53]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization defaults
sns.set_style('whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 150

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

In [54]:
df = pd.read_csv('../data/raw/archive_2/zameen.csv', sep='|')
print(f'Dataset: {df.shape[0]:,} listings \u00d7 {df.shape[1]} features')

Dataset: 16,044 listings × 6 features


## 2. Initial Exploration

### 2.1 Sample Listings

In [55]:
df.head(10)

,city,location,price,bedrooms,baths,size
0,Lahore,"DHA Phase 6, DHA Defence",74500000,5,6,"4,500.00"
1,Lahore,"DHA Phase 7, DHA Defence",51500000,5,6,"4,500.00"
2,Lahore,"Dream Gardens, Defence Road",7500000,1,1,518.00
3,Lahore,"DHA Phase 6, DHA Defence",73000000,5,6,"4,500.00"
4,Lahore,"Bahria Town - Sector B, Bahria Town",5700000,1,1,472.00
5,Lahore,"DHA Phase 5 - Block L, DHA Phase 5",53500000,5,6,"2,250.00"
6,Lahore,"Bahria Town - Overseas A, Bahria Town - Overse...",97500000,5,6,"4,500.00"
7,Lahore,"Bahria Town - Jasmine Block, Bahria Town - Sec...",47000000,5,7,"4,500.00"
8,Lahore,"Bahria Town - Sector E, Bahria Town",5000000,1,1,450.00
9,Lahore,"Raiwind Road, Lahore",8299000,1,1,630.00


### 2.2 Data Types & Memory

In [56]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16044 entries, 0 to 16043
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   city      16044 non-null  str    
 1   location  16044 non-null  str    
 2   price     16044 non-null  int64  
 3   bedrooms  16044 non-null  int64  
 4   baths     16044 non-null  int64  
 5   size      16044 non-null  float64
dtypes: float64(1), int64(3), str(2)
memory usage: 752.2 KB


### 2.3 Numerical Summary

In [57]:
df.describe()

,price,bedrooms,baths,size
count,"16,044.00","16,044.00","16,044.00","16,044.00"
mean,"45,711,134.19",3.78,4.06,"2,540.64"
std,"85,041,953.69",1.99,2.14,"10,607.07"
min,"70,000.00",0.00,0.00,0.00
25%,"14,000,000.00",3.00,3.00,"1,125.00"
50%,"24,000,000.00",4.00,4.00,"1,575.00"
75%,"45,000,000.00",5.00,6.00,"2,700.00"
max,"2,100,000,000.00",11.00,10.00,"1,215,000.00"


### 2.4 Categorical Summary

In [58]:
df.describe(include='object')

C:\Users\Pc\AppData\Local\Temp\ipykernel_38040\87514550.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include='object')


,city,location
count,16044,16044
unique,12,2038
top,Lahore,"DHA Villas, DHA Defence"
freq,2500,481


### 2.5 First Impressions

- **16,044 listings** across **12 cities** and **2,038 unique locations** — a broad snapshot of Pakistan's urban housing market
- **No missing values** in any column, though data quality issues exist (zero-size properties, extreme outliers)
- **Price range** spans 70,000 PKR to 2.1 Billion PKR — the high end likely includes commercial or incorrectly entered listings
- **Median price** (24M PKR) is roughly half the mean (45.7M PKR), indicating strong right-skew typical of real estate markets
- **Lahore dominates** with the highest listing count, followed by DHA/Bahria Town locations — Pakistan's premium housing societies
- **Size column** contains zero values and an extreme max of 1.2M sq ft — cleaning required before analysis

## 3. Data Quality Investigation

### 3.1 Missing Values Check

In [59]:
# Count missing values per column and show as both count and percentage
missing_count = df.isnull().sum()
missing_pct = df.isnull().mean() * 100

missing_summary = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

print('=== Missing Values Summary ===\n')
print(missing_summary.to_string())
print(f'\nTotal cells in dataset: {df.shape[0] * df.shape[1]:,}')
print(f'Total missing cells: {missing_count.sum():,}')

=== Missing Values Summary ===

          Missing Count  Missing %
city                  0       0.00
location              0       0.00
price                 0       0.00
bedrooms              0       0.00
baths                 0       0.00
size                  0       0.00

Total cells in dataset: 96,264
Total missing cells: 0


In [60]:
# Check for "hidden" missing values — zeros that likely represent missing data
print('=== Suspicious Zero Values ===\n')
for col in ['price', 'bedrooms', 'baths', 'size']:
    zero_count = (df[col] == 0).sum()
    zero_pct = (df[col] == 0).mean() * 100
    print(f'{col:>10}: {zero_count:,} zeros ({zero_pct:.2f}%)')

print('\n--- Interpretation ---')
print('A property with 0 size or 0 price is almost certainly a data entry error.')
print('Zero bedrooms could mean a studio/plot, but worth investigating.')

=== Suspicious Zero Values ===

     price: 0 zeros (0.00%)
  bedrooms: 1,620 zeros (10.10%)
     baths: 1,775 zeros (11.06%)
      size: 151 zeros (0.94%)

--- Interpretation ---
A property with 0 size or 0 price is almost certainly a data entry error.
Zero bedrooms could mean a studio/plot, but worth investigating.


In [61]:
# Are zero-bedroom and zero-bath listings the same rows?
both_zero = ((df['bedrooms'] == 0) & (df['baths'] == 0)).sum()
only_beds_zero = ((df['bedrooms'] == 0) & (df['baths'] != 0)).sum()
only_baths_zero = ((df['bedrooms'] != 0) & (df['baths'] == 0)).sum()

print('=== Zero Beds vs Zero Baths Overlap ===\n')
print(f'Both bedrooms AND baths = 0:  {both_zero:,}')
print(f'Only bedrooms = 0:            {only_beds_zero:,}')
print(f'Only baths = 0:               {only_baths_zero:,}')

print(f'\n--- Sample of 0-bedroom, 0-bath listings ---\n')
df[(df['bedrooms'] == 0) & (df['baths'] == 0)].sample(10, random_state=42)

=== Zero Beds vs Zero Baths Overlap ===

Both bedrooms AND baths = 0:  1,546
Only bedrooms = 0:            74
Only baths = 0:               229

--- Sample of 0-bedroom, 0-bath listings ---



,city,location,price,bedrooms,baths,size
14743,Murree,"Mall Road, Murree",13100000,0,0,"1,328.00"
9141,Multan,"Shah Rukn-e-Alam Colony - Block F, Shah Rukn-e...",4500000,0,0,675.00
12248,Rawalpindi,"Wakeel Colony, Islamabad Highway",7800000,0,0,900.00
9053,Multan,"Wapda Town Phase 1 - Block E, Wapda Town Phase 1",50000000,0,0,"4,500.00"
10728,Faisalabad,"Millat Town, Faisalabad",8000000,0,0,"1,125.00"
9864,Faisalabad,"Green Town, Faisalabad",15000000,0,0,"1,238.00"
15815,Gujranwala,"Rachna Pearl Hotel, GT Road",8198000,0,0,0.00
15017,Gujranwala,"Al-Mansoora Housing Society, Rahwali Cantt",15000000,0,0,"1,125.00"
11510,Peshawar,"Hayatabad Heights, Hayatabad Phase 3",12000000,0,0,0.00
11256,Peshawar,"Shoba Bazar, Peshawar",2392000,0,0,202.00


### 3.2 Column-by-Column Investigation

In [62]:
# City distribution — how many listings per city?
print('=== City Distribution ===\n')
city_counts = df['city'].value_counts()
print(city_counts)
print(f'\nTotal unique cities: {df["city"].nunique()}')

=== City Distribution ===

city
Lahore        2500
Karachi       2500
Rawalpindi    2500
Islamabad     2497
Multan        1923
Peshawar      1377
Faisalabad    1310
Gujranwala    1221
Murree         164
2_FECHS         25
Quetta          20
Attock           7
Name: count, dtype: int64

Total unique cities: 12


In [63]:
# Location — top 20 most common locations
print('=== Top 20 Locations ===\n')
print(df['location'].value_counts().head(20))
print(f'\nTotal unique locations: {df["location"].nunique()}')

=== Top 20 Locations ===

location
DHA Villas, DHA Defence                                                 481
Citi Housing Society, Gujranwala                                        453
Bahria Town Phase 8, Bahria Town Rawalpindi                             432
DHA Phase 6, DHA Defence                                                351
Warsak Road, Peshawar                                                   345
DHA Defence, Gujranwala                                                 343
Al-Ahad Heights, Defence Road                                           282
Buch Executive Villas, Multan                                           281
G-13, Islamabad                                                         220
Askari 5 - Sector J, Askari 5                                           205
Bahria Town Phase 8 - Ali Block, Bahria Town Phase 8 - Safari Valley    188
DHA Phase 7, DHA Defence                                                158
DHA Defence Phase 2, DHA Defence                     

In [64]:
# Bedrooms distribution
print('=== Bedrooms Distribution ===\n')
print(df['bedrooms'].value_counts().sort_index())
print(f'\nUnique values: {df["bedrooms"].nunique()}')

=== Bedrooms Distribution ===

bedrooms
0     1620
1      600
2     1413
3     3114
4     2776
5     3961
6     1664
7      523
8      208
9       73
10      67
11      25
Name: count, dtype: int64

Unique values: 12


In [65]:
# Baths distribution
print('=== Baths Distribution ===\n')
print(df['baths'].value_counts().sort_index())
print(f'\nUnique values: {df["baths"].nunique()}')

=== Baths Distribution ===

baths
0     1775
1      726
2     1232
3     1711
4     2788
5     2732
6     4093
7      657
8      241
9       34
10      55
Name: count, dtype: int64

Unique values: 11


In [66]:
# Price — key percentiles and extreme values
print('=== Price Investigation ===\n')
print(f'Min:    {df["price"].min():>20,.0f} PKR')
print(f'1st %:  {df["price"].quantile(0.01):>20,.0f} PKR')
print(f'5th %:  {df["price"].quantile(0.05):>20,.0f} PKR')
print(f'Median: {df["price"].median():>20,.0f} PKR')
print(f'95th %: {df["price"].quantile(0.95):>20,.0f} PKR')
print(f'99th %: {df["price"].quantile(0.99):>20,.0f} PKR')
print(f'Max:    {df["price"].max():>20,.0f} PKR')

print(f'\n--- Top 10 most expensive listings ---\n')
df.nlargest(10, 'price')[['city', 'location', 'price', 'bedrooms', 'size']]

=== Price Investigation ===

Min:                  70,000 PKR
1st %:             3,500,000 PKR
5th %:             6,500,000 PKR
Median:           24,000,000 PKR
95th %:          130,000,000 PKR
99th %:          460,000,000 PKR
Max:           2,100,000,000 PKR

--- Top 10 most expensive listings ---



,city,location,price,bedrooms,size
7186,Islamabad,"F-6/3, F-6",2100000000,5,"24,300.00"
739,Lahore,"Main Boulevard Gulberg, Gulberg",2000000000,11,"36,000.00"
5776,Islamabad,"F-6/2, F-6",1800000000,8,"18,000.00"
7483,Islamabad,"F-6, Islamabad",1650000000,7,"28,800.00"
6706,Islamabad,"F-7/2, F-7",1620000000,11,"18,000.00"
5701,Islamabad,"F-7/2, F-7",1600000000,0,"18,000.00"
6579,Islamabad,"F-6/2, F-6",1500000000,5,"22,500.00"
7130,Islamabad,"F-6, Islamabad",1350000000,9,"22,500.00"
7354,Islamabad,"F-6/3, F-6",1200000000,9,"18,000.00"
5648,Islamabad,"F-6, Islamabad",1100000000,4,"21,600.00"


In [67]:
# Size — key percentiles and extreme values
print('=== Size Investigation ===\n')
print(f'Min:    {df["size"].min():>15,.0f} sqft')
print(f'1st %:  {df["size"].quantile(0.01):>15,.0f} sqft')
print(f'5th %:  {df["size"].quantile(0.05):>15,.0f} sqft')
print(f'Median: {df["size"].median():>15,.0f} sqft')
print(f'95th %: {df["size"].quantile(0.95):>15,.0f} sqft')
print(f'99th %: {df["size"].quantile(0.99):>15,.0f} sqft')
print(f'Max:    {df["size"].max():>15,.0f} sqft')

print(f'\n--- Listings with 0 size ---\n')
print(f'Count: {(df["size"] == 0).sum()}')

print(f'\n--- Top 10 largest listings ---\n')
df.nlargest(10, 'size')[['city', 'location', 'price', 'bedrooms', 'size']]

=== Size Investigation ===

Min:                  0 sqft
1st %:              158 sqft
5th %:              603 sqft
Median:           1,575 sqft
95th %:           5,400 sqft
99th %:          12,150 sqft
Max:          1,215,000 sqft

--- Listings with 0 size ---

Count: 151

--- Top 10 largest listings ---



,city,location,price,bedrooms,size
16020,2_FECHS,"FECHS, E-11/2",80000000,7,"1,215,000.00"
828,Lahore,"Bedian Road, Lahore",450000000,1,"315,000.00"
6656,Islamabad,"Chak Shahzad Farms, Chak Shahzad",600000000,5,"180,000.00"
6693,Islamabad,"Chak Shahzad, Islamabad",330000000,5,"126,000.00"
7036,Islamabad,"Tarlai, Islamabad",360000000,7,"126,000.00"
6659,Islamabad,"Orchard Scheme, Islamabad",750000000,6,"94,500.00"
6660,Islamabad,"Orchard Scheme, Islamabad",750000000,6,"94,500.00"
6084,Islamabad,"Murree Road, Islamabad",600000000,5,"90,000.00"
6657,Islamabad,"Chak Shahzad, Islamabad",350000000,5,"90,000.00"
6658,Islamabad,"Chak Shahzad Farms, Chak Shahzad",350000000,5,"90,000.00"


### 3.3 Duplicate Check

In [68]:
# Check for exact duplicate rows
dup_count = df.duplicated().sum()
print(f'=== Duplicate Rows ===\n')
print(f'Exact duplicates: {dup_count:,} ({dup_count/len(df)*100:.2f}%)')

if dup_count > 0:
    print(f'\n--- Sample duplicate rows ---\n')
    # Show a few duplicated rows alongside their originals
    dup_mask = df.duplicated(keep=False)
    print(df[dup_mask].sort_values(by=['city', 'location', 'price']).head(10))

=== Duplicate Rows ===

Exact duplicates: 5,174 (32.25%)

--- Sample duplicate rows ---

             city                           location     price  bedrooms  \
16036     2_FECHS                      FECHS, E-11/2  60000000         8   
16042     2_FECHS                      FECHS, E-11/2  60000000         8   
16043     2_FECHS                      FECHS, E-11/2  60000000         8   
16034     2_FECHS                      FECHS, E-11/2  61000000         8   
16041     2_FECHS                      FECHS, E-11/2  61000000         8   
16030     2_FECHS                      FECHS, E-11/2  69000000         9   
16031     2_FECHS                      FECHS, E-11/2  69000000         9   
16037     2_FECHS                      FECHS, E-11/2  70000000         9   
16039     2_FECHS                      FECHS, E-11/2  70000000         9   
10476  Faisalabad  Abdullah Gardens, East Canal Road  45000000         5   

       baths     size  
16036      9 4,500.00  
16042      9 4,500.00  
16

### 3.4 Cardinality Summary

In [69]:
# Cardinality — how many unique values per column?
print('=== Cardinality (Unique Values per Column) ===\n')
for col in df.columns:
    n_unique = df[col].nunique()
    cardinality = 'LOW' if n_unique < 20 else ('MEDIUM' if n_unique < 100 else 'HIGH')
    print(f'{col:>10}: {n_unique:>6,} unique values  [{cardinality}]')

=== Cardinality (Unique Values per Column) ===

      city:     12 unique values  [LOW]
  location:  2,038 unique values  [HIGH]
     price:    948 unique values  [HIGH]
  bedrooms:     12 unique values  [LOW]
     baths:     11 unique values  [LOW]
      size:    349 unique values  [HIGH]


### 3.5 Data Quality Assessment

**Issues identified for cleaning:**

| # | Issue | Severity | Affected Rows | Cleaning Plan |
|---|-------|----------|---------------|---------------|
| 1 | **32% duplicate rows** (5,174 listings) | HIGH | 5,174 | Drop exact duplicates |
| 2 | **Dirty city name** — `2_FECHS` should be `Islamabad` | LOW | 25 | Remap to Islamabad |
| 3 | **Zero bedrooms/baths** — likely plots (no built structure) | MEDIUM | 1,546 (both zero) | Flag as "plot" property type, keep in dataset |
| 4 | **Zero size** — missing area data disguised as 0 | MEDIUM | 151 | Replace with NaN, decide impute vs drop later |
| 5 | **Extreme prices** — max 2.1B PKR vs median 24M | LOW | ~10 listings >1B | Verify against domain knowledge, cap or keep |
| 6 | **Extreme sizes** — max 1.2M sqft (28 acres) | LOW | ~5 listings >100K | Investigate, likely farmhouses or data errors |

**What's clean:**
- No formal null/NaN values in any column
- Price column is already numeric (no Lakh/Crore strings to parse)
- City names are consistent (except `2_FECHS`)
- Column names are clean and lowercase

## 4. Data Cleaning

### 4.1 Remove Duplicates

In [70]:
rows_before = len(df)
df = df.drop_duplicates()
rows_after = len(df)
rows_dropped = rows_before - rows_after

print(f'=== Duplicate Removal ===\n')
print(f'Before: {rows_before:,} rows')
print(f'After:  {rows_after:,} rows')
print(f'Dropped: {rows_dropped:,} rows ({rows_dropped/rows_before*100:.1f}%)')

# Reset index after dropping rows
df = df.reset_index(drop=True)
print(f'\nIndex reset: 0 to {len(df)-1}')

=== Duplicate Removal ===

Before: 16,044 rows
After:  10,870 rows
Dropped: 5,174 rows (32.2%)

Index reset: 0 to 10869


### 4.2 Fix City Names

In [71]:
# Fix dirty city name: 2_FECHS is a housing scheme in Islamabad (E-11 sector)
print('Before fix:')
print(df['city'].value_counts())

df['city'] = df['city'].replace('2_FECHS', 'Islamabad')

print(f'\nAfter fix:')
print(df['city'].value_counts())

Before fix:
city
Lahore        2149
Islamabad     1905
Karachi       1832
Rawalpindi    1460
Multan        1027
Faisalabad     965
Peshawar       913
Gujranwala     451
Murree         129
2_FECHS         20
Quetta          12
Attock           7
Name: count, dtype: int64

After fix:
city
Lahore        2149
Islamabad     1925
Karachi       1832
Rawalpindi    1460
Multan        1027
Faisalabad     965
Peshawar       913
Gujranwala     451
Murree         129
Quetta          12
Attock           7
Name: count, dtype: int64


In [72]:
# Strip whitespace from string columns to prevent invisible mismatches
for col in ['city', 'location']:
    before = df[col].nunique()
    df[col] = df[col].str.strip()
    after = df[col].nunique()
    diff = before - after
    print(f'{col}: {before} unique -> {after} unique (merged {diff} whitespace duplicates)')

city: 11 unique -> 11 unique (merged 0 whitespace duplicates)
location: 2038 unique -> 2038 unique (merged 0 whitespace duplicates)


### 4.3 Convert Data Types

In [73]:
# Record memory usage before conversion
mem_before = df.memory_usage(deep=True).sum() / 1024
print(f'Memory before: {mem_before:,.1f} KB\n')
print('Current dtypes:')
print(df.dtypes)

Memory before: 1,803.7 KB

Current dtypes:
city            str
location        str
price         int64
bedrooms      int64
baths         int64
size        float64
dtype: object


In [74]:
# Convert low-cardinality columns to category dtype
df['city'] = df['city'].astype('category')
df['bedrooms'] = df['bedrooms'].astype('category')
df['baths'] = df['baths'].astype('category')

# Replace zero sizes with NaN (not a real measurement)
df['size'] = df['size'].replace(0, np.nan)

mem_after = df.memory_usage(deep=True).sum() / 1024
savings = (1 - mem_after / mem_before) * 100

print(f'Memory after:  {mem_after:,.1f} KB')
print(f'Savings:       {savings:.1f}%\n')
print('Updated dtypes:')
print(df.dtypes)

Memory after:  1,062.2 KB
Savings:       41.1%

Updated dtypes:
city        category
location         str
price          int64
bedrooms    category
baths       category
size         float64
dtype: object


### 4.4 Cleaning Verification

In [75]:
# Final check after cleaning steps
print('=== Post-Cleaning Summary ===\n')
print(f'Shape: {df.shape[0]:,} rows \u00d7 {df.shape[1]} columns')
print(f'Duplicates remaining: {df.duplicated().sum()}')
print(f'Unique cities: {df["city"].nunique()} (was 12, now {df["city"].nunique()} after merging 2_FECHS)')
print(f'Null values in size: {df["size"].isnull().sum()} (converted from zero)')
print(f'\n--- City distribution (cleaned) ---\n')
print(df['city'].value_counts())

print(f'\n--- Data types ---\n')
print(df.dtypes)

print(f'\n--- Quick stats ---')
df.describe()

=== Post-Cleaning Summary ===

Shape: 10,870 rows × 6 columns
Duplicates remaining: 0
Unique cities: 11 (was 12, now 11 after merging 2_FECHS)
Null values in size: 76 (converted from zero)

--- City distribution (cleaned) ---

city
Lahore        2149
Islamabad     1925
Karachi       1832
Rawalpindi    1460
Multan        1027
Faisalabad     965
Peshawar       913
Gujranwala     451
Murree         129
Quetta          12
Attock           7
Name: count, dtype: int64

--- Data types ---

city        category
location         str
price          int64
bedrooms    category
baths       category
size         float64
dtype: object

--- Quick stats ---


,price,size
count,"10,870.00","10,794.00"
mean,"50,419,882.34","2,773.05"
std,"93,075,386.33","12,785.21"
min,"70,000.00",45.00
25%,"14,000,000.00","1,125.00"
50%,"26,000,000.00","1,800.00"
75%,"50,000,000.00","2,799.00"
max,"2,100,000,000.00","1,215,000.00"


### 4.5 Cleaning Summary

| Step | Action | Impact |
|------|--------|--------|
| Duplicates | Dropped exact duplicate rows | ~5,174 rows removed (32%) |
| City fix | Remapped `2_FECHS` to `Islamabad` | 25 rows corrected, 12 \u2192 11 cities |
| Whitespace | Stripped leading/trailing spaces from text columns | Prevented invisible mismatches |
| Data types | Converted `city`, `bedrooms`, `baths` to `category` | Reduced memory usage |
| Zero sizes | Replaced 0 values in `size` with NaN | 151 rows now have explicit missing size |

### 4.6 Price Column — Validation & Cleaning

Pakistani real estate prices use a traditional unit system:

| Unit | Value | Example |
|------|-------|---------|
| **Lakh** | 100,000 PKR | "45 Lakh" = 4,500,000 PKR |
| **Crore** | 10,000,000 PKR | "1.2 Crore" = 12,000,000 PKR |
| **Arab** | 1,000,000,000 PKR | "1 Arab" = 1,000,000,000 PKR |

The price column in this dataset is already stored as numeric values in PKR. The steps below validate the values and add a human-readable `price_lakh` column for easier interpretation.

In [76]:
# Re-examine price distribution after deduplication
print('=== Price Column — Post-Cleaning Stats ===\n')
print(f'Data type: {df["price"].dtype}')
print(f'Non-null:  {df["price"].notna().sum():,}')
print(f'Zeros:     {(df["price"] == 0).sum()}')
print(f'Negatives: {(df["price"] < 0).sum()}\n')

# Key percentiles
percentiles = [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
print(f'{"Percentile":>12} {"PKR":>20} {"In Lakh":>12}')
print('-' * 46)
for p in percentiles:
    val = df['price'].quantile(p)
    print(f'{p*100:>10.0f}th  {val:>20,.0f} {val/100_000:>12,.1f}')
print(f'{"Min":>12} {df["price"].min():>20,.0f} {df["price"].min()/100_000:>12,.1f}')
print(f'{"Max":>12} {df["price"].max():>20,.0f} {df["price"].max()/100_000:>12,.1f}')

print(f'\nMean:   {df["price"].mean():>20,.0f} PKR')
print(f'Median: {df["price"].median():>20,.0f} PKR')
print(f'Skew:   {df["price"].skew():>20,.2f}')

=== Price Column — Post-Cleaning Stats ===

Data type: int64
Non-null:  10,870
Zeros:     0
Negatives: 0

  Percentile                  PKR      In Lakh
----------------------------------------------
         1th             3,500,000         35.0
         5th             6,182,650         61.8
        25th            14,000,000        140.0
        50th            26,000,000        260.0
        75th            50,000,000        500.0
        95th           155,000,000      1,550.0
        99th           480,000,000      4,800.0
         Min               70,000          0.7
         Max        2,100,000,000     21,000.0

Mean:             50,419,882 PKR
Median:           26,000,000 PKR
Skew:                   7.92


In [77]:
# Investigate suspicious extremes using domain knowledge
print('=== Suspiciously Low Prices (< 5 Lakh / 500K PKR) ===\n')
low_price = df[df['price'] < 500_000]
print(f'Count: {len(low_price)}')
if len(low_price) > 0:
    print(low_price[['city', 'location', 'price', 'bedrooms', 'size']].to_string())

print(f'\n=== Suspiciously High Prices (> 100 Crore / 1B PKR) ===\n')
high_price = df[df['price'] > 1_000_000_000]
print(f'Count: {len(high_price)}')
if len(high_price) > 0:
    print(high_price[['city', 'location', 'price', 'bedrooms', 'size']].to_string())

print(f'\n=== Price Sanity Bands ===\n')
bands = [
    ('< 5 Lakh (likely error)', df['price'] < 500_000),
    ('5 Lakh – 50 Lakh', (df['price'] >= 500_000) & (df['price'] < 5_000_000)),
    ('50 Lakh – 5 Crore', (df['price'] >= 5_000_000) & (df['price'] < 50_000_000)),
    ('5 Crore – 50 Crore', (df['price'] >= 50_000_000) & (df['price'] < 500_000_000)),
    ('50 Crore – 100 Crore', (df['price'] >= 500_000_000) & (df['price'] <= 1_000_000_000)),
    ('> 100 Crore (likely error)', df['price'] > 1_000_000_000),
]
for label, mask in bands:
    count = mask.sum()
    pct = count / len(df) * 100
    print(f'{label:>30}: {count:>5,} listings ({pct:>5.1f}%)')

=== Suspiciously Low Prices (< 5 Lakh / 500K PKR) ===

Count: 3
           city                          location   price bedrooms     size
68       Lahore          DHA Phase 1, DHA Defence   70000        3 4,500.00
4078  Islamabad  DHA Defence Phase 2, DHA Defence  110000        3 4,500.00
4103  Islamabad  DHA Defence Phase 2, DHA Defence   75000        3 3,600.00

=== Suspiciously High Prices (> 100 Crore / 1B PKR) ===

Count: 11
           city                         location       price bedrooms      size
676      Lahore  Main Boulevard Gulberg, Gulberg  2000000000       11 36,000.00
4559  Islamabad                   F-6, Islamabad  1100000000        4 21,600.00
4600  Islamabad                       F-7/2, F-7  1600000000        0 18,000.00
4659  Islamabad                       F-6/2, F-6  1100000000        4 21,600.00
4660  Islamabad                       F-6/2, F-6  1800000000        8 18,000.00
5277  Islamabad                       F-6/2, F-6  1500000000        5 22,500.00
5360

In [78]:
# Flag price outliers for later review (not removing yet — that's Session 6)
# Using IQR method as a statistical reference point
Q1 = df['price'].quantile(0.25)
Q3 = df['price'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print('=== IQR-Based Outlier Bounds ===\n')
print(f'Q1 (25th):     {Q1:>20,.0f} PKR')
print(f'Q3 (75th):     {Q3:>20,.0f} PKR')
print(f'IQR:           {IQR:>20,.0f} PKR')
print(f'Lower bound:   {lower_bound:>20,.0f} PKR')
print(f'Upper bound:   {upper_bound:>20,.0f} PKR')

below = (df['price'] < lower_bound).sum()
above = (df['price'] > upper_bound).sum()
print(f'\nBelow lower bound: {below:,} listings')
print(f'Above upper bound: {above:,} listings')
print(f'Total flagged:     {below + above:,} ({(below + above)/len(df)*100:.1f}%)')

print('\nNote: IQR flags are aggressive on skewed data — many "outliers" above')
print('the upper bound are legitimate luxury properties. Domain knowledge will')
print('guide final outlier removal in Session 6.')

=== IQR-Based Outlier Bounds ===

Q1 (25th):               14,000,000 PKR
Q3 (75th):               50,000,000 PKR
IQR:                     36,000,000 PKR
Lower bound:            -40,000,000 PKR
Upper bound:            104,000,000 PKR

Below lower bound: 0 listings
Above upper bound: 1,016 listings
Total flagged:     1,016 (9.3%)

Note: IQR flags are aggressive on skewed data — many "outliers" above
the upper bound are legitimate luxury properties. Domain knowledge will
guide final outlier removal in Session 6.


In [79]:
# Create human-readable price column (vectorized — fast)
df['price_lakh'] = df['price'] / 100_000

print('=== New Column: price_lakh ===\n')
print(df[['city', 'location', 'price', 'price_lakh']].sample(10, random_state=42).to_string())
print(f'\nMedian price: {df["price_lakh"].median():,.1f} Lakh PKR')
print(f'Mean price:   {df["price_lakh"].mean():,.1f} Lakh PKR')

=== New Column: price_lakh ===

             city                                     location     price  price_lakh
2754      Karachi                     Creek Vista, DHA Phase 8  69500000      695.00
10155  Rawalpindi  Bahria Town Phase 8, Bahria Town Rawalpindi  56500000      565.00
8289     Peshawar                       Dalazak Road, Peshawar  23000000      230.00
6186       Multan                        Askari 3, DHA Defence  26000000      260.00
9792   Rawalpindi  Bahria Town Phase 7, Bahria Town Rawalpindi  12200000      122.00
6879       Multan                           Akbar Road, Multan  26500000      265.00
9935   Rawalpindi  Gulbahar Scheme - Sector 1, Gulbahar Scheme   5100000       51.00
5547    Islamabad                              E-11, Islamabad  16500000      165.00
1149       Lahore                     DHA Phase 8, DHA Defence  61500000      615.00
7704   Faisalabad                   Jaranwala Road, Faisalabad   4500000       45.00

Median price: 260.0 Lakh PKR
Mea

In [80]:
# DEMONSTRATION: What if prices were messy strings?
# This function is NOT needed for our data, but shows how apply() works
# with custom parsing logic — a common real-world scenario.

def parse_price_string(price_str):
    """Convert strings like '1.2 Crore' or '45 Lakh' to numeric PKR."""
    try:
        text = str(price_str).strip().lower().replace(',', '')
        
        if 'arab' in text:
            return float(text.replace('arab', '').strip()) * 1_000_000_000
        elif 'crore' in text:
            return float(text.replace('crore', '').strip()) * 10_000_000
        elif 'lakh' in text:
            return float(text.replace('lakh', '').strip()) * 100_000
        else:
            return float(text)
    except (ValueError, AttributeError):
        return np.nan

# Test on synthetic examples to prove it works
test_cases = ['1.2 Crore', '45 Lakh', '2 Arab', '5,000,000', 'N/A', '  3.5 crore  ']
print('=== parse_price_string() Demo ===\n')
print(f'{"Input":>20} {"Output (PKR)":>20}')
print('-' * 42)
for tc in test_cases:
    result = parse_price_string(tc)
    if pd.isna(result):
        print(f'{tc:>20} {"NaN":>20}')
    else:
        print(f'{tc:>20} {result:>20,.0f}')

print('\nThis function would be applied with: df["price"] = df["price_raw"].apply(parse_price_string)')
print('Not needed here since our prices are already numeric.')

=== parse_price_string() Demo ===

               Input         Output (PKR)
------------------------------------------
           1.2 Crore           12,000,000
             45 Lakh            4,500,000
              2 Arab        2,000,000,000
           5,000,000            5,000,000
                 N/A                  NaN
         3.5 crore             35,000,000

This function would be applied with: df["price"] = df["price_raw"].apply(parse_price_string)
Not needed here since our prices are already numeric.


In [81]:
# Final price column verification
print('=== Price Column — Final State ===\n')
print(f'Column "price":      {df["price"].dtype}, {df["price"].notna().sum():,} non-null')
print(f'Column "price_lakh": {df["price_lakh"].dtype}, {df["price_lakh"].notna().sum():,} non-null')
print(f'\nPrice range: {df["price_lakh"].min():,.1f} Lakh — {df["price_lakh"].max():,.1f} Lakh')
print(f'             ({df["price_lakh"].min()/100:,.2f} Crore — {df["price_lakh"].max()/100:,.2f} Crore)')
print(f'\nColumns in dataframe: {list(df.columns)}')
print(f'Shape: {df.shape}')

=== Price Column — Final State ===

Column "price":      int64, 10,870 non-null
Column "price_lakh": float64, 10,870 non-null

Price range: 0.7 Lakh — 21,000.0 Lakh
             (0.01 Crore — 210.00 Crore)

Columns in dataframe: ['city', 'location', 'price', 'bedrooms', 'baths', 'size', 'price_lakh']
Shape: (10870, 7)


### 4.7 Price Cleaning Summary

**Findings:**
- Price column was already numeric — no string parsing required
- No zero or negative prices detected in the dataset
- Distribution is heavily right-skewed: a small number of ultra-premium listings (>100 Crore) inflate the mean well above the median
- IQR method flags a significant portion of upper-end listings, but many are legitimate luxury properties in DHA/Bahria Town

**Actions taken:**
- Created `price_lakh` column for human-readable price interpretation
- Documented IQR bounds for reference in outlier removal (Session 6)

**Deferred to Session 6:**
- Final outlier removal decisions (domain knowledge + statistical methods combined)
- Determining whether extreme prices (>100 Crore) should be capped, removed, or kept